# **FRAMEWORK V7: NOTEBOOK DE EXTRACCIÓN DE DATOS CRUDOS**

CAPA - CALIDAD DE AGUA

## **M0. Configuración General**

In [ ]:
import pandas as pd
import requests
import io

print("Configuración general cargada.")

Configuración general cargada.


## **M1. Definición de la Fuente Oficial**

In [ ]:
api_id = "62gv-3857"
url_api = f"https://www.datos.gov.co/resource/{api_id}.json"

print(f"API ID: {api_id}")
print(f"URL de la API: {url_api}")

API ID: 62gv-3857
URL de la API: https://www.datos.gov.co/resource/62gv-3857.json


## **M2. Extracción y Procesamiento de Datos**

In [ ]:
print("Iniciando extracción y filtrado semántico de la Cuenca del Río Bogotá...")

try:
    # Solicitamos una carga mayor (150,000 registros) para asegurar capturar la zona central
    params = {'$limit': 150000}
    response = requests.get(url_api, params=params)

    if response.status_code == 200:
        df_completo = pd.DataFrame(response.json())

        # 1. Limpieza de nombres de columnas (eliminamos espacios ocultos o caracteres raros)
        df_completo.columns = df_completo.columns.str.strip()

        # 2. Identificación dinámica de columnas cortadas (por si cambian los nombres)
        col_corriente = [c for c in df_completo.columns if 'corriente' in c][0]
        col_municipio = [c for c in df_completo.columns if 'municipio' in c][0]
        col_parametro = [c for c in df_completo.columns if 'obse' in c or 'para' in c][0] # Halla 'edad_obse'

        # 3. Aplicación de Filtro Espacial: Buscamos "BOGOTA" en la corriente hídrica
        df_bogota = df_completo[df_completo[col_corriente].str.upper().str.contains("BOGOTA", na=False)]

        print("\n¡Filtrado por Cuenca Completado!")
        print(f"Registros encontrados específicos del Río Bogotá: {df_bogota.shape[0]}")
        print("-" * 75)

        if df_bogota.shape[0] > 0:
            # Mostramos los municipios y parámetros disponibles en el Río Bogotá
            print("Municipios detectados con estaciones activas:")
            print(df_bogota[col_municipio].unique())
            print("\nParámetros fisicoquímicos disponibles en la serie:")
            print(df_bogota[col_parametro].unique()[:15]) # Muestra los primeros 15 parámetros
            print("-" * 75)

            # Exportamos la data pura del Río Bogotá a Excel
            df_bogota.to_excel("Calidad_Agua_Rio_Bogota_V1.xlsx", index=False)
            print("Archivo 'Calidad_Agua_Rio_Bogota_V1.xlsx' generado con éxito.")
        else:
            print("Alerta: El lote de datos analizado no contenía registros de la cuenca de Bogotá.")
            print("Sugerencia: Ampliar el '$limit' en los parámetros para buscar más atrás en el histórico.")

    else:
        print(f"Error de conexión. Código: {response.status_code}")

except Exception as e:
    print(f"Error técnico durante el filtrado: {e}")

Iniciando extracción y filtrado semántico de la Cuenca del Río Bogotá...

¡Filtrado por Cuenca Completado!
Registros encontrados específicos del Río Bogotá: 10875
---------------------------------------------------------------------------
Municipios detectados con estaciones activas:
['BOGOTÁ D.C.' 'CHÍA' 'COTA' 'EL COLEGIO' 'GIRARDOT' 'SOACHA' 'TOCAIMA'
 'TOCANCIPÁ' 'VILLAPINZÓN']

Parámetros fisicoquímicos disponibles en la serie:
['CONDUCTIVIDAD ELECTRICA' 'DEMANDA QUIMICA DE OXIGENO (DQO)'
 'FOSFORO TOTAL' 'NITROGENO TOTAL' 'OXIGENO DISUELTO (OD)' 'pH'
 'SOLIDOS SUSPENDIDOS TOTALES' 'TEMPERATURA' 'CADMIO TOTAL EN AGUA'
 'COBRE TOTAL EN AGUA' 'CROMO TOTAL EN AGUA' 'FOSFORO REACTIVO DISUELTO'
 'NIQUEL TOTAL EN AGUA' 'NITRATO' 'NITRITO']
---------------------------------------------------------------------------
Archivo 'Calidad_Agua_Rio_Bogota_V1.xlsx' generado con éxito.


In [ ]:
# Cargar el dataset inicial desde el archivo local
df = pd.read_excel("Calidad_Agua_Rio_Bogota_V1.xlsx")

# 2. Selección de los 10 parámetros más frecuentes
top_10_params = df['propiedad_observada'].value_counts().nlargest(10).index.tolist()
df_filtrado = df[df['propiedad_observada'].isin(top_10_params)].copy()

# Convertir la columna 'resultado' a numérica, convirtiendo los errores a NaN
df_filtrado['resultado'] = pd.to_numeric(df_filtrado['resultado'], errors='coerce')

# 3. Conversión de fecha y Remuestreo Mensual
df_filtrado['fecha'] = pd.to_datetime(df_filtrado['fecha'])
# Pivotamos para que cada parámetro sea una columna
df_pivot = df_filtrado.pivot_table(index='fecha',
                                   columns='propiedad_observada',
                                   values='resultado',
                                   aggfunc='mean')

# Remuestreo a frecuencia mensual
df_mensual = df_pivot.resample('ME').mean() # Usar 'ME' para fin de mes

# 4. Limpieza: Interpolación lineal para cerrar brechas temporales
df_mensual = df_mensual.interpolate(method='linear').fillna(method='bfill').fillna(method='ffill') # Asegurar que no queden nulos al principio

# 5. Exportación del dataset transformado
df_mensual.to_excel("01_Capa_Calidad_Agua_V1.xlsx")

print("--- TRANSFORMACIÓN COMPLETADA ---")
print(f"Dimensiones del dataset mensual: {df_mensual.shape}")
print(f"Parámetros seleccionados: {df_mensual.columns.tolist()}")
print("Archivo '01_Capa_Calidad_Agua_V1.xlsx' generado con éxito.")

--- TRANSFORMACIÓN COMPLETADA ---
Dimensiones del dataset mensual: (233, 10)
Parámetros seleccionados: ['CONDUCTIVIDAD ELECTRICA', 'DEMANDA QUIMICA DE OXIGENO (DQO)', 'FOSFORO REACTIVO DISUELTO', 'NITRATO', 'NITRITO', 'OXIGENO DISUELTO (OD)', 'SOLIDOS SUSPENDIDOS TOTALES', 'TEMPERATURA', 'TURBIDEZ', 'pH']
Archivo '01_Capa_Calidad_Agua_V1.xlsx' generado con éxito.


/tmp/ipykernel_1265/3282809693.py:23: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_mensual = df_mensual.interpolate(method='linear').fillna(method='bfill').fillna(method='ffill') # Asegurar que no queden nulos al principio


## **M3. Reporte de Auditoría**

In [ ]:
# Auditoría del archivo inicial `Calidad_Agua_Rio_Bogota_V1.xlsx`
df_calidad_initial = pd.read_excel("Calidad_Agua_Rio_Bogota_V1.xlsx")

print("--- AUDITORÍA DE CALIDAD DE AGUA RÍO BOGOTÁ V1 (Inicial) ---")

# 2. Verificación de estructura y dimensiones
print(f"Dimensiones del dataset: {df_calidad_initial.shape}")
print(f"Columnas disponibles: {list(df_calidad_initial.columns)}")

# 3. Auditoría de Parámetros (¿Qué estamos midiendo?)
col_parametro_initial = [c for c in df_calidad_initial.columns if 'obse' in c or 'para' in c][0]
col_fecha_initial = [c for c in df_calidad_initial.columns if 'fecha' in c][0]

print("\n--- Distribución de Parámetros (Top 10) ---")
print(df_calidad_initial[col_parametro_initial].value_counts().head(10))

# 4. Auditoría de nulos
print(f"\nTotal de registros nulos: {df_calidad_initial.isnull().sum().sum()}")

# 5. Rango Temporal
df_calidad_initial[col_fecha_initial] = pd.to_datetime(df_calidad_initial[col_fecha_initial])
print(f"\nRango temporal: desde {df_calidad_initial[col_fecha_initial].min()} hasta {df_calidad_initial[col_fecha_initial].max()}")

--- AUDITORÍA DE CALIDAD DE AGUA RÍO BOGOTÁ V1 (Inicial) ---
Dimensiones del dataset: (10875, 16)
Columnas disponibles: ['nombre_del_punto_de_monitoreo', 'latitud', 'longitud', 'elevaci_n_m_s_n_m', 'corriente', 'zona_hidrogr_fica_zh', 'szh_c_digo_rea_zona_subzona', 'nombre_subzona_hidrogr_fica', 'departamento', 'municipio', 'fecha', 'propiedad_observada', 'resultado', 'unidad_del_resultado', 'proyecto', 'codigo__muestra']

--- Distribución de Parámetros (Top 10) ---
propiedad_observada
CONDUCTIVIDAD ELECTRICA             474
pH                                  474
TEMPERATURA                         474
SOLIDOS SUSPENDIDOS TOTALES         473
OXIGENO DISUELTO (OD)               471
DEMANDA QUIMICA DE OXIGENO (DQO)    468
FOSFORO REACTIVO DISUELTO           419
TURBIDEZ                            417
NITRITO                             415
NITRATO                             408
Name: count, dtype: int64

Total de registros nulos: 0

Rango temporal: desde 2005-02-15 00:00:00 hasta 2024-

In [ ]:
# Función de auditoría adaptada para archivos locales
def auditar_dataset_local(file_path, col_fecha):
    try:
        df = pd.read_excel(file_path)

        df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')
        df_clean = df.dropna(subset=[col_fecha]).copy()

        df_clean['year'] = df_clean[col_fecha].dt.year
        conteo_por_año = df_clean['year'].value_counts().sort_index()

        print(f"--- Auditoría: {file_path} ---")
        print(f"Rango temporal: {df_clean[col_fecha].min()} a {df_clean[col_fecha].max()}")
        print("\nRegistros por año:")
        print(conteo_por_año)
        print("-" * 40)

        return df_clean, conteo_por_año

    except Exception as e:
        print(f"Error al procesar el archivo '{file_path}': {e}")
        return pd.DataFrame(), pd.Series()

df_auditoria_initial, conteo_anual_initial = auditar_dataset_local("Calidad_Agua_Rio_Bogota_V1.xlsx", 'fecha')

--- Auditoría: Calidad_Agua_Rio_Bogota_V1.xlsx ---
Rango temporal: 2005-02-15 00:00:00 a 2024-06-19 00:00:00

Registros por año:
year
2005    630
2006    668
2007    666
2008    614
2009    623
2010    655
2011    519
2012    486
2013    549
2014    396
2015    565
2016    552
2017    406
2018    453
2019    545
2020    173
2021    845
2022    635
2023    642
2024    253
Name: count, dtype: int64
----------------------------------------


In [ ]:
# Listar los valores únicos de la columna propiedad_observada del dataset inicial
df_initial_params = pd.read_excel("Calidad_Agua_Rio_Bogota_V1.xlsx")
valores_unicos = df_initial_params['propiedad_observada'].unique()
print("Variables disponibles en 'propiedad_observada' del dataset inicial:")
for v in valores_unicos:
    print(f"- {v}")

Variables disponibles en 'propiedad_observada' del dataset inicial:
- CONDUCTIVIDAD ELECTRICA
- DEMANDA QUIMICA DE OXIGENO (DQO)
- FOSFORO TOTAL
- NITROGENO TOTAL
- OXIGENO DISUELTO (OD)
- pH
- SOLIDOS SUSPENDIDOS TOTALES
- TEMPERATURA
- CADMIO TOTAL EN AGUA
- COBRE TOTAL EN AGUA
- CROMO TOTAL EN AGUA
- FOSFORO REACTIVO DISUELTO
- NIQUEL TOTAL EN AGUA
- NITRATO
- NITRITO
- NITROGENO AMONIACAL
- PLOMO TOTAL EN AGUA
- SULFATO
- TURBIDEZ
- ZINC TOTAL EN AGUA
- ALUMINIO POTENCIALMENTE BIODISPONIBLE
- ALUMINIO TOTAL EN AGUA
- CADMIO POTENCIALMENTE BIODISPONIBLE
- CAUDAL
- COBRE POTENCIALMENTE BIODISPONIBLE
- CROMO POTENCIALMENTE BIODISPONIBLE
- DEMANDA BIOQUIMICA DE OXIGENO (DBO5)
- HIERRO POTENCIALMENTE BIODISPONIBLE
- HIERRO TOTAL EN AGUA
- MANGANESO POTENCIALMENTE BIODISPONIBLE
- MANGANESO TOTAL EN AGUA
- NIQUEL POTENCIALMENTE BIODISPONIBLE
- NITROGENO KJELDAHL TOTAL
- PLOMO POTENCIALMENTE BIODISPONIBLE
- ZINC POTENCIALMENTE BIODISPONIBLE
- CARBONO ORGANICO TOTAL (COT)
- SOLIDOS TOTALES


Esta auditoría revela un dataset extraordinariamente rico, pero también extremadamente complejo. tenemos una base de datos de 10,875 registros que abarca casi dos décadas (2005-2024), con una enorme cantidad de parámetros fisicoquímicos y microbiológicos.

**Análisis de la Auditoría**

* Robustez Histórica: La serie temporal es muy sólida. A pesar de una baja en 2020 (probablemente por restricciones de campo debido a la pandemia), la cantidad de datos anuales es suficiente para soportar un análisis de tendencias multivariado para el modelo.

* Heterogeneidad de Parámetros: Tenemos desde variables físicas básicas (pH, Conductividad, Temperatura) hasta contaminantes complejos (metales pesados, pesticidas, hidrocarburos).

    * El reto: Esta es una matriz de datos dispersa. No todos los parámetros se miden con la misma frecuencia ni en todos los puntos de monitoreo.

* Consistencia: El hecho de tener 0 nulos indica que la estructura de la base de datos es muy limpia, lo cual es una ventaja operativa crítica.

* Potencial Analítico: Tienes variables que son indicadores directos de la salud del río, como DBO5 (Demanda Bioquímica de Oxígeno) y OD (Oxígeno Disuelto), que son las que mejor correlacionan con la calidad de agua y el impacto antrópico que seguramente quieres capturar en tu tesis.

***¿Es necesario seguir investigando el archivo?***

Sí, absolutamente, pero con un enfoque de "Selección Estratégica" antes de integrarlo.

Si intentamos mantener las más de 60 variables que aparecen en propiedad_observada al modelo, vamo a sufrir un fenómeno llamado "maldición de la dimensionalidad" y una pérdida masiva de datos, porque al hacer el pivot (tabla dinámica para alinear fechas), muchas filas quedarán vacías donde no se midió un parámetro específico.

**El plan de acción recomendado:**

* Filtrado por "Parámetros Clave": No necesitas todos los pesticidas para el modelo global. Te sugiero filtrar el dataset para quedarte solo con los parámetros con mayor frecuencia de muestreo (aquellos que vimos en el top 10: pH, Conductividad, DBO, DQO, OD, Sólidos, etc.).

* Agregación Espacial: Debemos decidir si vas a usar el promedio mensual de toda la cuenca o si vas a seleccionar puntos de monitoreo estratégicos (ej. aguas arriba y aguas abajo de Bogotá).

* Remuestreo Mensual: Al igual que con los niveles del río, esta serie debe consolidarse a nivel mensual para que pueda ser "fucionada".

In [ ]:
# Auditoría del archivo final `01_Capa_Calidad_Agua_V1.xlsx`

try:
    df_final_audit = pd.read_excel("01_Capa_Calidad_Agua_V1.xlsx", index_col='fecha')

    print("--- AUDITORÍA DE CALIDAD DE AGUA (TOP 10 MENSUAL) ---")

    # 1. Verificación de dimensiones
    print(f"Dimensiones (Meses, Parámetros): {df_final_audit.shape}")

    # 2. Verificación de nulos
    nulos = df_final_audit.isnull().sum().sum()
    print(f"Total de valores nulos encontrados: {nulos}")

    # 3. Verificación de rango temporal
    print(f"Rango temporal: {df_final_audit.index.min()} a {df_final_audit.index.max()}")

    # 4. Estadísticas básicas para verificar escalas
    print("\n--- Primeras 5 filas del Dataset ---")
    print(df_final_audit.head())

    # 5. Validación final para fusión
    if nulos == 0 and df_final_audit.shape[1] == 10:
        print("\nEstado: AUDITORÍA PASADA (Dataset optimizado para integración)")
    else:
        print("\nEstado: ALERTA (Revisar estructura o nulos remanentes)")

except Exception as e:
    print(f"Error durante la auditoría del dataset final: {e}")

--- AUDITORÍA DE CALIDAD DE AGUA (TOP 10 MENSUAL) ---
Dimensiones (Meses, Parámetros): (233, 10)
Total de valores nulos encontrados: 0
Rango temporal: 2005-02-28 00:00:00 a 2024-06-30 00:00:00

--- Primeras 5 filas del Dataset ---
            CONDUCTIVIDAD ELECTRICA  DEMANDA QUIMICA DE OXIGENO (DQO)  \
fecha                                                                   
2005-02-28               448.483333                        100.500000   
2005-03-31               412.252593                        118.740741   
2005-04-30               376.021852                        136.981481   
2005-05-31               339.791111                        155.222222   
2005-06-30               375.705000                        147.383333   

            FOSFORO REACTIVO DISUELTO   NITRATO   NITRITO  \
fecha                                                       
2005-02-28                   0.540000  0.878750  0.147389   
2005-03-31                   0.465696  0.892352  0.136314   
2005-04-30   

## **M4. Exportación de Resultados**

In [ ]:
# El archivo '01_Capa_Calidad_Agua_V1.xlsx' ya fue generado en la etapa M2.
# Ahora generaremos un archivo de auditoría con el conteo de registros por año del dataset inicial.

if not conteo_anual_initial.empty:
    df_auditoria_export = conteo_anual_initial.reset_index()
    df_auditoria_export.columns = ['Año', 'Cantidad_Registros']
    df_auditoria_export.to_excel("Auditoria_Capa_Calidad.xlsx", index=False)
    print("Archivo 'Auditoria_Capa_Calidad.xlsx' generado con el conteo anual de registros iniciales.")
else:
    print("No se pudo generar 'Auditoria_Capa_Calidad.xlsx' porque no hay datos de auditoría anual.")

Archivo 'Auditoria_Capa_Calidad.xlsx' generado con el conteo anual de registros iniciales.


## **M5. Resumen Final**

El cuaderno ha sido reorganizado siguiendo la estructura solicitada, extrayendo datos, procesándolos a nivel mensual y realizando auditorías detalladas. Se han generado dos archivos Excel clave: `01_Capa_Calidad_Agua_V1.xlsx` con la data de calidad del agua transformada y `Auditoria_Capa_Calidad.xlsx` con un resumen del conteo anual de registros del dataset inicial, listos para su integración y análisis posterior. Este resumen detalla el trabajo realizado en los datos a lo largo de este cuaderno, **destacando que, dependiendo del proceso y los requisitos del modelo, se pueden utilizar tanto el dataset inicial de más de 10,000 registros como los datos filtrados y transformados para construir la capa final.**